# EcoSort Waste Management Assistant
# Module 8 Summative Lab

## Overview

You are a data scientist at "EcoSort," a technology company that specializes in developing AI solutions for waste management. EcoSort has partnered with Metro City's waste management department to develop an intelligent waste management assistant that can help residents properly dispose of waste items so less time is spent sorting material at facilities.

This assistant needs to:

1. Identify waste materials from images uploaded by residents (CNN)
2. Classify waste items based on text descriptions provided by residents (RNN/Transformer)
3. Generate specific recycling instructions based on identified waste type and city policies (Generative Transformer with RAG)

Your task is to build this integrated system using the RealWaste dataset along with generated text data that simulates real-world waste management operations.

## Part 1: Dataset Exploration and Preparation

In this section, you will explore and prepare the datasets for your models.

### 1.1 Load and Explore the RealWaste Dataset

In [3]:
# Import necessary libraries
import random
import numpy as np
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Hides INFO and WARNING logs from TensorFlow
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import pathlib

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [ ]:
# TODO: Load and explore the RealWaste dataset
# - Dataset structure
# - Distribution of waste categories
# - Image characteristics (resolution, quality, background)

# Your code here


# --- STEP 1: Unzip RealWaste Dataset if not already extracted ---
import zipfile
from pathlib import Path

zip_path = Path('realwaste.zip')
extract_path = Path('RealWaste')

if zip_path.exists() and not extract_path.exists():
    print(f"Extracting '{zip_path}' into '{extract_path}'...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
elif extract_path.exists():
    print("Verified: 'RealWaste' directory already exists.")
else:
    print("Note: 'realwaste.zip' not found, proceeding with existing directories.")

# Load and explore the RealWaste dataset structure, categories, and image characteristics
import pathlib
import tensorflow as tf

# Define data directory
data_dir = pathlib.Path('RealWaste')

# 1. Dataset structure and category distribution
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
num_classes = len(class_names)
print(f"Dataset Structure Path: {data_dir.absolute()}")
print(f"Number of Waste Categories: {num_classes}")
print(f"Categories: {class_names}\n")

print("Distribution of Waste Categories:")
total_images = 0
for class_name in class_names:
    class_path = data_dir / class_name
    image_count = len(list(class_path.glob('*.*')))
    total_images += image_count
    print(f" - {class_name}: {image_count} images")

print(f"\nTotal Images across all categories: {total_images}")

# 2. Image characteristics inspection (resolution, quality, background)
sample_image_path = next(data_dir.glob('*/*.*'))
# Load a sample image using tf.keras.utils to inspect shape/resolution
sample_img = tf.keras.utils.load_img(sample_image_path)
sample_array = tf.keras.utils.img_to_array(sample_img)

print("\nImage Characteristics:")
print(f" - Sample Image File: {sample_image_path.name}")
print(f" - Original Resolution (Height, Width, Channels): {sample_array.shape}")
print(f" - Quality & Background: Images are captured in authentic landfill reception environments with varied unadulterated forms and background noise.")

Verified: 'RealWaste' directory already exists.
Dataset Structure Path: /home/komen/Documents/moringa_school_projects/Module_5_labs/dsc-ecosort-nn-summative-lab/RealWaste
Number of Waste Categories: 9
Categories: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']

Distribution of Waste Categories:
 - Cardboard: 461 images
 - Food Organics: 411 images
 - Glass: 420 images
 - Metal: 790 images
 - Miscellaneous Trash: 495 images
 - Paper: 500 images
 - Plastic: 921 images
 - Textile Trash: 318 images
 - Vegetation: 436 images

Total Images across all categories: 4752

Image Characteristics:
 - Sample Image File: Miscellaneous Trash_252.jpg
 - Original Resolution (Height, Width, Channels): (524, 524, 3)
 - Quality & Background: Images are captured in authentic landfill reception environments with varied unadulterated forms and background noise.


### 1.2 Explore Text Datasets

In [14]:
# TODO: Load and explore the waste description text data
# - Load waste_descriptions.csv
# - Analyze vocabulary and structure
# - Understand the distribution of categories

# Your code here
# Load and explore the waste description text data
import pandas as pd

# 1. Load waste_descriptions.csv
descriptions_df = pd.read_csv("waste_descriptions.csv")

print("--- Dataset Shape & Structure ---")
print(f"Total Rows and Columns: {descriptions_df.shape}")
print(f"Column Names: {descriptions_df.columns.tolist()}\n")

print("--- First 3 Rows Preview ---")
display(descriptions_df.head(3))

print("\n--- Missing Values Analysis ---")
print(descriptions_df.isnull().sum())

print("\n--- Distribution of Categories ---")
category_counts = descriptions_df['category'].value_counts()
print(category_counts)

# 2. Analyze vocabulary and text structure
# Calculate average description length (word count and character count)
descriptions_df["word_count"] = descriptions_df["description"].apply(lambda x: len(str(x).split()))
descriptions_df["char_count"] = descriptions_df["description"].apply(lambda x: len(str(x)))

print("\n--- Vocabulary & Text Structure Statistics ---")
print(f"Average Word Count per Description: {descriptions_df['word_count'].mean():.2f}")
print(f"Average Character Count per Description: {descriptions_df['char_count'].mean():.2f}")
print(f"Total Unique Vocabulary Words (approximate): {len(set(' '.join(descriptions_df['description'].astype(str)).lower().split()))}")

--- Dataset Shape & Structure ---
Total Rows and Columns: (5000, 5)
Column Names: ['description', 'category', 'disposal_instruction', 'common_confusion', 'material_composition']

--- First 3 Rows Preview ---


,description,category,disposal_instruction,common_confusion,material_composition
0,soiled silver tablecloth,Textile Trash,Look for textile recycling programs in your area.,NaN,"Fabric made from natural or synthetic fibers, ..."
1,folded glass bottle leaking,Glass,"Remove caps, lids, and corks before recycling.",NaN,"Silica-based material, may contain additives f..."
2,large Supermarket vegetable waste with food re...,Food Organics,"If no compost available, place in general waste.",NaN,Biodegradable matter derived from plant or ani...



--- Missing Values Analysis ---
description                0
category                   0
disposal_instruction       0
common_confusion        2504
material_composition       0
dtype: int64

--- Distribution of Categories ---
category
Vegetation             600
Textile Trash          586
Cardboard              584
Miscellaneous Trash    578
Plastic                569
Glass                  551
Food Organics          518
Metal                  508
Paper                  506
Name: count, dtype: int64

--- Vocabulary & Text Structure Statistics ---
Average Word Count per Description: 4.85
Average Character Count per Description: 33.41
Total Unique Vocabulary Words (approximate): 305


In [9]:
# TODO: Load and explore the waste policy documents
# - Load waste_policy_documents.csv
# - Understand document organization and language

# Your code here
# Load and explore the waste policy documents
import json

# 1. Load waste_policy_documents.json
with open("waste_policy_documents.json", "r") as f:
    policy_documents = json.load(f)

print(f"--- Document Organization & Overview ---")
print(f"Total Policy Documents Loaded: {len(policy_documents)}")

# 2. Analyze structure and metadata
print(f"\nExample Document Keys / Fields: {list(policy_documents[0].keys())}")

# Collect unique policy types and jurisdictions
policy_types = [doc["policy_type"] for doc in policy_documents]
jurisdictions = set(doc["jurisdiction"] for doc in policy_documents)

print(f"Jurisdictions Covered: {jurisdictions}")
print(f"Total Unique Policy Types: {len(set(policy_types))}")

print("\n--- Language & Content Structure Sample ---")
# Display text style and organization of the first policy document
print(f"Policy Type: {policy_documents[0]['policy_type']}")
print(f"Effective Date: {policy_documents[0]['effective_date']}")
print(f"Categories Covered: {policy_documents[0]['categories_covered']}")
print("-" * 40)
print(policy_documents[0]["document_text"][:300] + "...\n[Content truncated for display]")

--- Document Organization & Overview ---
Total Policy Documents Loaded: 14

Example Document Keys / Fields: ['policy_id', 'policy_type', 'categories_covered', 'effective_date', 'document_text', 'jurisdiction']
Jurisdictions Covered: {'Metro City'}
Total Unique Policy Types: 14

--- Language & Content Structure Sample ---
Policy Type: Textile Trash Recycling Guidelines
Effective Date: 2023-11-04
Categories Covered: ['Textile Trash']
----------------------------------------
TEXTILE RECYCLING GUIDELINES

Acceptable Items:
- Clean clothing (all conditions)
- Towels, sheets, and linens
- Fabric scraps
- Curtains and cloth napkins
- Handbags and backpacks made of fabric
- Soft toys and stuffed animals

Non-Acceptable Items:
- Wet or moldy textiles
- Heavily soiled items
- ...
[Content truncated for display]


### 1.3 Create Data Pipelines

In [10]:
# Run this code to setup the images properly into train, validation, and test sets
# Set your data directory path - update this with your actual path
import pathlib
data_dir = pathlib.Path('RealWaste')

# Parameters
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

# Calculate the total number of classes automatically from the directory structure
num_classes = len([item for item in data_dir.glob('*') if item.is_dir()])
print(f"Number of classes: {num_classes}")

# List all class folders
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print(f"Class names: {class_names}")

# Count all images
image_count = len(list(data_dir.glob('*/*.jpg'))) + len(list(data_dir.glob('*/*.png')))
print(f"Total images found: {image_count}")

# Create a dataset using tf.keras.utils.image_dataset_from_directory
# This will automatically split the data into training and validation sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # For one-hot encoded labels
    shuffle=True
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # For one-hot encoded labels
    shuffle=True
)

# Create a separate test dataset by taking part of the validation set
# First, let's get the number of batches in the validation set
val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

print(f"Number of training batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Number of validation batches: {tf.data.experimental.cardinality(validation_ds)}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dataset)}")

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Number of classes: 9
Class names: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
Total images found: 4752
Found 4752 files belonging to 9 classes.
Using 3802 files for training.
Found 4752 files belonging to 9 classes.
Using 950 files for validation.


E0000 00:00:1793270641.468654   50554 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1793270641.469060   51215 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1793270641.488946   50554 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Number of training batches: 119
Number of validation batches: 15
Number of test batches: 15


In [13]:
# TODO: Create a text preprocessing pipeline
# - Tokenization
# - Text cleaning
# - Split data into train and test
# - Create embeddings/features

# Your code here
# 1. Text Cleaning: We convert all characters to lowercase and strip out punctuation
#    and special characters so that words like "Plastic", "plastic", and "plastic!"
#    are treated as the exact same token by our machine learning model.
# 2. Train/Test Split: We partition our descriptions into 80% training data and
#    20% testing data using stratification (`stratify=y`) to ensure every waste
#    category maintains a balanced representation in both sets.
# 3. Tokenization & Feature Extraction (TF-IDF): We use Term Frequency-Inverse
#    Document Frequency (TF-IDF) to convert raw text into numerical feature vectors.
#    This measures how important a word is to a document relative to the entire corpus,
#    while automatically removing common English stop words (like "the", "and", "is").

import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Step A: Load the waste descriptions dataset
df_text = pd.read_csv('waste_descriptions.csv')

# Step B: Define text cleaning function
def clean_text(text):
    """Normalizes text: lowercase, removes special characters, fixes whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Strip punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Clean excess spaces
    return text

# Apply cleaning function to create the processed text column
df_text['cleaned_description'] = df_text['description'].apply(clean_text)

# Step C: Split data into training and testing sets (80/20 ratio with stratification)
X = df_text['cleaned_description']
y = df_text['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples count: {len(X_train)}")
print(f"Testing samples count: {len(X_test)}")

# Step D: Create TF-IDF vectorizer embeddings/features
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit on training data and transform both train and test sets
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"TF-IDF Training Feature Matrix Shape: {X_train_vectorized.shape}")
print("Text preprocessing and embedding pipeline successfully executed!")

Training samples count: 4000
Testing samples count: 1000
TF-IDF Training Feature Matrix Shape: (4000, 298)
Text preprocessing and embedding pipeline successfully executed!


In [15]:
# TODO: Prepare documents for RAG
# - Document preprocessing
# - Create embeddings for retrieval

# Your code here

# 1. Document Preprocessing & Chunking: Policy documents in JSON format contain
#    rich multi-sentence rules. We parse these documents and split (chunk) them into
#    smaller, digestible text segments so our retrieval system can pinpoint exact
#    rules rather than searching through massive blocks of text.
# 2. Embedding Generation (TF-IDF / Vector Space): We convert these text chunks
#    into numerical vector representations using TF-IDF vectorization. This transforms
#    words into mathematical coordinates, allowing us to compute similarity scores
#    between a user's recycling question and our policy document database.

import json
from sklearn.feature_extraction.text import TfidfVectorizer

# Step A: Load the waste policy documents JSON file
with open('waste_policy_documents.json', 'r') as f:
    policy_data = json.load(f)

print(f"Loaded {len(policy_data)} raw policy documents from JSON.")

# Step B: Document Preprocessing & Chunking
# Extract document texts, policy types, and metadata into a list of chunks
policy_chunks = []
metadata_store = []

for doc in policy_data:
    doc_id = doc.get('policy_id', 'unknown')
    policy_type = doc.get('policy_type', 'General Policy')
    jurisdiction = doc.get('jurisdiction', 'Metro City')
    text = doc.get('document_text', '')

    # Simple sentence-based or paragraph-based chunking
    # Here we split large document texts into individual paragraphs/sentences for precise retrieval
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 20]

    for i, sentence in enumerate(sentences):
        chunk_text = f"Policy: {policy_type} | Jurisdiction: {jurisdiction} | Rule: {sentence}."
        policy_chunks.append(chunk_text)
        metadata_store.append({
            'policy_id': doc_id,
            'policy_type': policy_type,
            'jurisdiction': jurisdiction,
            'chunk_index': i
        })

print(f"Generated {len(policy_chunks)} searchable policy chunks after preprocessing.")

# Step C: Create Embeddings for Retrieval using TF-IDF
rag_vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')

# Fit and transform the policy chunks into a searchable vector space matrix
policy_embeddings = rag_vectorizer.fit_transform(policy_chunks)

print(f"RAG Embedding Matrix Shape: {policy_embeddings.shape}")
print("RAG document preparation and embedding pipeline successfully completed!")


Loaded 14 raw policy documents from JSON.
Generated 27 searchable policy chunks after preprocessing.
RAG Embedding Matrix Shape: (27, 295)
RAG document preparation and embedding pipeline successfully completed!


## Part 2: Waste Material Classification with CNN

In this section, you will build a CNN model to classify waste materials from images.

### 2.1 Preprocess Images

In [ ]:
# TODO: Implement image preprocessing
# - Apply the preprocessing pipeline created earlier

# Your code here

### 2.2 Implement CNN Model with Transfer Learning

In [ ]:
# TODO: Select an appropriate base model and implement transfer learning
# - Choose from MobileNet, EfficientNet, etc.
# - Add custom classification layers for the 9 waste categories
# - Configure loss function and metrics

# Your code here

### 2.3 Train and Evaluate the Model

In [ ]:
# TODO: Train the CNN model
# - Use appropriate batch size and epochs
# - Implement regularization to prevent overfitting
# - Monitor training and validation metrics

# Your code here

In [ ]:
# TODO: Evaluate model performance
# - Calculate accuracy on test set
# - Generate confusion matrix
# - Analyze error patterns

# Your code here

### 2.4 Fine-tune the Model

In [ ]:
# TODO: Tune model parameters to improve performance
# - Adjust learning rate
# - Add regularization, dropout
# - Modify architecture if needed

# Your code here

## Part 3: Waste Description Classification

In this section, you will build a text classification model to categorize waste based on descriptions.

### 3.1 Preprocess Text Data

In [ ]:
# TODO: Implement text preprocessing
# - Apply the text preprocessing pipeline created earlier

# Your code here

### 3.2 Implement Text Classification Model

In [ ]:
# TODO: Choose and implement a text classification model
# Option A: Traditional ML model (Naive Bayes, Random Forest, etc.)
# Option B: Fine-tune a transformer-based model (BERT, DistilBERT, etc.)

# Your code here

### 3.3 Train and Evaluate the Model

In [ ]:
# TODO: Train the text classification model
# - Use appropriate training parameters
# - Monitor training progress

# Your code here

In [ ]:
# TODO: Evaluate model performance
# - Calculate accuracy on test set
# - Generate confusion matrix
# - Analyze error patterns

# Your code here

### 3.4 Create Classification Function

In [ ]:
# TODO: Create a function that takes a text description and returns the predicted waste category

def classify_waste_description(description):
    """
    Classifies a waste description into an appropriate category.

    Args:
        description (str): Text description of waste item

    Returns:
        str: Predicted waste category
    """
    # Your code here
    pass

## Part 4: Recycling Instruction Generation with RAG

In this section, you will implement a Retrieval-Augmented Generation (RAG) system to generate recycling instructions.

### 4.1 Preprocess Documents for Retrieval

In [ ]:
# TODO: Prepare documents for retrieval
# - Process policy documents and disposal instructions
# - Create embeddings for efficient retrieval

# Your code here

### 4.2 Implement RAG-based System

In [ ]:
# TODO: Select a pre-trained language model and implement RAG
# - Choose an appropriate language model
# - Create a retrieval mechanism

# Your code here

### 4.3 Adjust and Evaluate the System

In [ ]:
# TODO: Train the RAG-based system
# - Adjust sampling methods/parameters

# Your code here

In [ ]:
# TODO: Evaluate the quality of generated instructions
# - Test with various waste categories
# - Assess relevance and accuracy

# Your code here

### 4.4 Create Instruction Generation Function

In [ ]:
# TODO: Create a function that takes a waste category and generates recycling instructions

def generate_recycling_instructions(waste_category):
    """
    Generates detailed recycling instructions for a given waste category.

    Args:
        waste_category (str): Waste category

    Returns:
        str: Detailed recycling instructions
        list: Relevant policy documents
    """
    # Your code here
    pass

## Part 5: Integrated Waste Management Assistant

In this section, you will integrate all three models into a unified waste management assistant.

### 5.1 Design Integration Architecture

In [ ]:
# TODO: Design an architecture that integrates all three models
# - Create interfaces between components
# - Handle input/output flow

# Your code here

### 5.2 Implement Integrated Assistant

In [ ]:
# TODO: Implement the integrated waste management assistant

def waste_management_assistant(input_data, input_type="image"):
    """
    Integrated waste management assistant that processes either images or text descriptions
    and returns waste classification and recycling instructions.

    Args:
        input_data: Either an image file path/array or a text description
        input_type (str): Type of input - "image" or "text"

    Returns:
        dict: Dictionary containing waste category, confidence, and recycling instructions
    """
    # Your code here
    pass

### 5.3 Evaluate the Integrated System

In [ ]:
# TODO: Evaluate the integrated system on test cases
# - Test with images from test dataset
# - Test with text descriptions from test dataset
# - Assess overall performance

# Your code here

## Submission Guidelines

1. Make sure all code cells are properly commented and annotated
2. Ensure that all functions are implemented and working correctly
3. Verify that all evaluation metrics are calculated and analyzed
4. Double-check that the integrated system works as expected
5. Submit your completed and annotated Jupyter notebook file

Remember to demonstrate your understanding of the underlying concepts and provide justification for your design decisions throughout the notebook.